In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import torch
print(torch.cuda.is_available())

In [ ]:
!git clone https://github.com/jugal-sac/IR-colorization-BAH2026.git
%cd IR-colorization-BAH2026
!ls

In [ ]:
!pip install rasterio numpy opencv-python matplotlib tifffile scikit-image

In [ ]:
!pip install earthengine-api geemap

In [ ]:
import rasterio, numpy, cv2, matplotlib, tifffile, skimage, ee, geemap
from PIL import Image
print("All dependencies confirmed working")

In [ ]:
!grep -n "PROJECT_ID\|ee_project_id" scripts/download_data.sh


In [ ]:
!sed -i 's/YOUR_ACTUAL_PROJECT_ID/hackathonisro2026/g' scripts/download_data.sh
!cat scripts/download_data.sh | grep -i project

In [ ]:
import geemap
help(geemap.ee_export_image)

In [ ]:
!grep -n "filename_prefix" scripts/download.py

In [ ]:
!grep -n "filename_prefix\|ee_export_image" scripts/download.py

In [ ]:
file_path = 'scripts/download.py'

with open(file_path, 'r') as f:
    content = f.read()

old_line = "geemap.ee_export_image(image, output_path, scale=30, region=image.geometry().bounds(), file_per_band=True, filename_prefix=filename_prefix)"
new_line = "geemap.ee_export_image(image, filename=output_path, scale=30, region=image.geometry().bounds(), file_per_band=True)"

if old_line in content:
    content = content.replace(old_line, new_line)
    with open(file_path, 'w') as f:
        f.write(content)
    print("✅ Line replaced successfully")
else:
    print("⚠️ Exact line not found — paste the grep output so we match it precisely")

In [ ]:
!grep -n "ee_export_image" scripts/download.py

In [ ]:
!grep -n "filename_prefix" scripts/download.py

In [ ]:
!./scripts/download_data.sh

In [ ]:
import ee

# This forces the link to appear inside the notebook output
ee.Authenticate(auth_mode='notebook')


In [ ]:
import ee
ee.Initialize(project='hackathonisro2026')
print(ee.String('Connected!').getInfo())

In [ ]:
# Fix the hardcoded placeholder project ID in the script
!sed -i 's/your-google-earth-engine-project-id/hackathonisro2026/g' \
    scripts/download_data.sh

# Run it
!chmod +x scripts/download_data.sh
!./scripts/download_data.sh

In [ ]:
import ee
import os

ee.Initialize(project='hackathonisro2026')

# Create output folder
os.makedirs('input/demo_product', exist_ok=True)

# Load Landsat 9 - pick a scene over India
image = ee.ImageCollection('LANDSAT/LC09/C02/T1') \
    .filterDate('2024-01-01', '2024-03-31') \
    .filterBounds(ee.Geometry.Point([77.2090, 28.6139])) \
    .sort('CLOUD_COVER') \
    .first()

print('Scene found:', image.get('LANDSAT_PRODUCT_ID').getInfo())

In [ ]:
import geemap

# Define area (Delhi region - change if needed)
aoi = ee.Geometry.Rectangle([76.8, 28.4, 77.6, 29.0])

bands = {
    'B2': 'input/demo_product/scene_B2.tif',
    'B3': 'input/demo_product/scene_B3.tif',
    'B4': 'input/demo_product/scene_B4.tif',
    'B10': 'input/demo_product/scene_B10.tif',
}

for band, path in bands.items():
    print(f'Downloading {band}...')
    geemap.ee_export_image(
        image.select([band]),
        filename=path,
        scale=30,
        region=aoi,
        file_per_band=False
    )
    print(f'✅ {band} saved to {path}')

In [ ]:
import rasterio

for band, path in bands.items():
    with rasterio.open(path) as src:
        print(f'✅ {band} | Resolution: {src.res} | Shape: {src.shape}')

In [ ]:
!pip install rasterio tifffile opencv-python-headless \
    scikit-image numpy pillow --quiet
print("✅ Dependencies installed")

In [ ]:
import os
os.chdir('/content/IR-colorization-BAH2026')
!ls  # confirm driver.py is here

In [ ]:
import os, shutil

# driver.py looks for files ending in _B2.TIF _B3.TIF _B4.TIF _B10.TIF
# Your files are named scene_B2.tif etc — need to rename

src_dir = 'input/demo_product/'
files = os.listdir(src_dir)
print("Current files:", files)

for f in files:
    if f.startswith('scene_'):
        # scene_B2.tif → demo_product_B2.TIF
        band = f.split('_')[1].replace('.tif', '')
        new_name = f'demo_scene_{band}.TIF'
        os.rename(os.path.join(src_dir, f),
                  os.path.join(src_dir, new_name))
        print(f"Renamed: {f} → {new_name}")

print("\nFiles after rename:")
print(os.listdir(src_dir))

In [ ]:
import os
os.chdir('/content/IR-colorization-BAH2026')
print("Current dir:", os.getcwd())
print("Contents:", os.listdir('.'))

In [ ]:
import os
print("Contents of /content:", os.listdir('/content'))

In [ ]:
import os
os.chdir('/content')

!git clone https://github.com/DarkCipher-z/bah2026-ps10-team.git IR-colorization-BAH2026

if os.path.isdir('/content/IR-colorization-BAH2026'):
    os.chdir('/content/IR-colorization-BAH2026')
    print("✅ Now in:", os.getcwd())
    print("Contents:", os.listdir('.'))
else:
    print("❌ Clone failed - check repo URL")

In [ ]:
import os
print(os.listdir('/content/drive/MyDrive/BAH2026'))

In [ ]:
!find /content/drive/MyDrive -iname "*.git" -maxdepth 5 2>/dev/null

In [ ]:
import shutil
shutil.copytree('/content/drive/MyDrive/BAH2026/input/', '/content/IR-colorization-BAH2026/input/', dirs_exist_ok=True)
shutil.copytree('/content/drive/MyDrive/BAH2026/output/', '/content/IR-colorization-BAH2026/output/', dirs_exist_ok=True)
print("✅ input/ and output/ restored from Drive")

In [ ]:
import os

if os.path.exists('input/demo_product/'):
    print("✅ Found:", os.listdir('input/demo_product/'))
else:
    print("❌ Folder missing - session reset, need to re-download")

In [ ]:
# Re-download bands from Earth Engine
import ee, geemap, os

ee.Initialize(project='hackathonisro2026')
os.makedirs('input/demo_product/', exist_ok=True)

image = ee.ImageCollection('LANDSAT/LC09/C02/T1') \
    .filterDate('2024-01-01', '2024-03-31') \
    .filterBounds(ee.Geometry.Point([77.2090, 28.6139])) \
    .sort('CLOUD_COVER').first()

aoi = ee.Geometry.Rectangle([76.8, 28.4, 77.6, 29.0])

for band, path in {
    'B2': 'input/demo_product/scene_B2.TIF',
    'B3': 'input/demo_product/scene_B3.TIF',
    'B4': 'input/demo_product/scene_B4.TIF',
    'B10': 'input/demo_product/scene_B10.TIF'
}.items():
    print(f'Downloading {band}...')
    geemap.ee_export_image(
        image.select([band]),
        filename=path, scale=30,
        region=aoi, file_per_band=False
    )
    print(f'✅ {band} done')

In [ ]:
import os
os.chdir('/content/IR-colorization-BAH2026')
!ls  # confirm driver.py is here

In [ ]:
import os

src_dir = 'input/demo_product/'
files = os.listdir(src_dir)
print("Before rename:", files)

for f in files:
    if f.startswith('scene_') and f.endswith('.TIF'):
        # scene_B2.TIF → demo_scene_B2.TIF
        band = f.replace('scene_', '').replace('.TIF', '')
        new_name = f'demo_scene_{band}.TIF'
        os.rename(
            os.path.join(src_dir, f),
            os.path.join(src_dir, new_name)
        )
        print(f"✅ {f} → {new_name}")

print("\nAfter rename:", os.listdir(src_dir))

In [ ]:
!pip install tifffile opencv-python-headless \
    scikit-image pillow --quiet
print("✅ Done")

In [ ]:
!python driver.py

In [ ]:
import os

patch_dir = 'output/patches/'
if os.path.exists(patch_dir):
    files = os.listdir(patch_dir)
    npy = [f for f in files if f.endswith('.npy')]
    png = [f for f in files if f.endswith('.png')]
    print(f"✅ .npy patches: {len(npy)}")
    print(f"📸 .png previews: {len(png)}")
    print("Sample:", files[:4])
else:
    print("❌ No patches folder - check driver.py output above")

In [ ]:
import os

# Walk all subfolders to find patches
for root, dirs, files in os.walk('output/'):
    for f in files:
        print(os.path.join(root, f))

In [ ]:
import shutil, os

base = '/content/drive/MyDrive/BAH2026'

# Save patches
shutil.copytree(
    'output/',
    f'{base}/output/',
    dirs_exist_ok=True
)
# Save raw bands
shutil.copytree(
    'input/',
    f'{base}/input/',
    dirs_exist_ok=True
)
print("✅ Everything saved to Drive!")

In [ ]:
import ee, geemap, os

ee.Initialize(project='hackathonisro2026')

# Different locations across India for varied terrain
scenes_config = [
    ([72.8, 18.9, 73.2, 19.3], 'mumbai_coast'),
    ([77.5, 12.8, 78.0, 13.2], 'bengaluru_urban'),
    ([88.2, 22.4, 88.6, 22.8], 'kolkata_river'),
    ([78.0, 26.0, 78.5, 26.5], 'gwalior_rural'),
    ([76.5, 30.5, 77.0, 31.0], 'chandigarh_mixed'),
]

for coords, name in scenes_config:
    print(f"\n📥 Downloading {name}...")
    os.makedirs(f'input/{name}/', exist_ok=True)

    aoi = ee.Geometry.Rectangle(coords)
    image = ee.ImageCollection('LANDSAT/LC09/C02/T1') \
        .filterDate('2024-01-01', '2024-06-30') \
        .filterBounds(aoi) \
        .sort('CLOUD_COVER') \
        .first()

    for band in ['B2', 'B3', 'B4', 'B10']:
        path = f'input/{name}/scene_{band}.TIF'
        geemap.ee_export_image(
            image.select([band]),
            filename=path,
            scale=30,
            region=aoi,
            file_per_band=False
        )
    print(f"✅ {name} done!")

print("\n🎉 All scenes downloaded!")

In [ ]:
import os

input_dir = 'input/'
for scene in os.listdir(input_dir):
    scene_path = os.path.join(input_dir, scene)
    if not os.path.isdir(scene_path):
        continue
    for f in os.listdir(scene_path):
        if f.startswith('scene_') and \
           (f.endswith('.TIF') or f.endswith('.tif')):
            band = f.replace('scene_', '') \
                    .replace('.tif','').replace('.TIF','')
            new_name = f'{scene}_{band}.TIF'
            os.rename(
                os.path.join(scene_path, f),
                os.path.join(scene_path, new_name)
            )
            print(f"✅ {f} → {new_name}")

print("\nAll scenes ready!")

In [ ]:
!python driver.py

In [ ]:
import os

total_npy = 0
total_png = 0

for root, dirs, files in os.walk('output/patches/'):
    npy = [f for f in files if f.endswith('.npy')]
    png = [f for f in files if f.endswith('.png')]
    if npy or png:
        print(f"📁 {root}: {len(npy)} npy, {len(png)} png")
    total_npy += len(npy)
    total_png += len(png)

print(f"\n{'='*40}")
print(f"Total .npy patches: {total_npy}")
print(f"Total .png previews: {total_png}")
if total_npy >= 100:
    print("✅ Enough patches to start training!")
else:
    print("⚠️ Need more scenes — add more in Step 3")

In [ ]:
import ee, geemap, os

ee.Initialize(project='hackathonisro2026')

# MUCH larger areas (1.5° × 1.5°) = ~165km × 165km
# This will give hundreds of patches per scene
scenes_config = [
    ([74.0, 17.5, 75.5, 19.0], 'maharashtra'),
    ([76.5, 11.5, 78.0, 13.0], 'karnataka'),
    ([86.5, 21.5, 88.0, 23.0], 'westbengal'),
    ([76.5, 29.0, 78.0, 30.5], 'haryana'),
    ([79.5, 25.0, 81.0, 26.5], 'uttarpradesh'),
    ([72.5, 22.5, 74.0, 24.0], 'gujarat'),
]

for coords, name in scenes_config:
    print(f"\n📥 Downloading {name}...")
    os.makedirs(f'input/{name}/', exist_ok=True)

    aoi = ee.Geometry.Rectangle(coords)

    # Get least cloudy scene
    image = ee.ImageCollection('LANDSAT/LC09/C02/T1') \
        .filterDate('2024-01-01', '2024-04-30') \
        .filterBounds(aoi) \
        .sort('CLOUD_COVER') \
        .first()

    try:
        pid = image.get('LANDSAT_PRODUCT_ID').getInfo()
        print(f"  Scene: {pid}")
    except:
        print(f"  ⚠️ No scene found for {name}, skipping")
        continue

    for band in ['B2', 'B3', 'B4', 'B10']:
        path = f'input/{name}/{name}_{band}.TIF'
        if os.path.exists(path):
            print(f"  ⏭️ {band} already exists")
            continue
        geemap.ee_export_image(
            image.select([band]),
            filename=path,
            scale=30,
            region=aoi,
            file_per_band=False
        )
        print(f"  ✅ {band} done")

print("\n🎉 All scenes downloaded!")

In [ ]:
import shutil, os

# Clear old tiny patches
shutil.rmtree('output/', ignore_errors=True)
os.makedirs('output/', exist_ok=True)

print("✅ Old output cleared")
print("Now running driver.py...")
!python driver.py

In [ ]:
import os

total_npy = 0
for root, dirs, files in os.walk('output/patches/'):
    npy = [f for f in files if f.endswith('.npy')]
    if npy:
        print(f"📁 {os.path.basename(root)}: {len(npy)} patches")
    total_npy += len(npy)

print(f"\n{'='*40}")
print(f"Total .npy patches: {total_npy}")
if total_npy >= 500:
    print("✅ Good enough to start training!")
elif total_npy >= 100:
    print("⚠️ Decent start - can begin smoke test")
else:
    print("❌ Still too few - areas too small")

In [ ]:
import os
import rasterio

print("Checking downscaled image sizes:")
downscale_dir = 'output/downscaled_data/'

for f in os.listdir(downscale_dir):
    if f.endswith('.tif'):
        with rasterio.open(os.path.join(downscale_dir, f)) as src:
            print(f"{f}: {src.width}x{src.height} pixels")

In [ ]:
import ee, geemap, os

ee.Initialize(project='hackathonisro2026')

# Very large area - entire Rajasthan region
# 5° × 5° = ~550km × 550km - will give thousands of patches
large_scenes = [
    ([72.0, 24.0, 77.0, 29.0], 'rajasthan'),   # 5°×5°
    ([77.0, 8.0,  82.0, 13.0], 'tamilnadu'),    # 5°×5°
]

for coords, name in large_scenes:
    print(f"\n📥 Downloading {name} (large area)...")
    os.makedirs(f'input/{name}/', exist_ok=True)

    aoi = ee.Geometry.Rectangle(coords)

    image = ee.ImageCollection('LANDSAT/LC09/C02/T1') \
        .filterDate('2024-01-01', '2024-06-30') \
        .filterBounds(aoi) \
        .sort('CLOUD_COVER') \
        .first()

    try:
        pid = image.get('LANDSAT_PRODUCT_ID').getInfo()
        print(f"  Scene ID: {pid}")
    except:
        print(f"  ⚠️ No scene for {name}")
        continue

    for band in ['B2', 'B3', 'B4', 'B10']:
        path = f'input/{name}/{name}_{band}.TIF'
        print(f"  Downloading {band}...")
        geemap.ee_export_image(
            image.select([band]),
            filename=path,
            scale=30,        # 30m native resolution
            region=aoi,
            file_per_band=False
        )
        print(f"  ✅ {band} done")

print("\n🎉 Large scene download complete!")

In [ ]:
import ee, geemap, os

ee.Initialize(project='hackathonisro2026')

# Split large area into smaller 1°×1° tiles at 100m scale
# This stays under the 50MB limit
scenes = [
    ([72.0, 24.0, 73.0, 25.0], 'raj_tile1'),
    ([73.0, 24.0, 74.0, 25.0], 'raj_tile2'),
    ([74.0, 24.0, 75.0, 25.0], 'raj_tile3'),
    ([72.0, 25.0, 73.0, 26.0], 'raj_tile4'),
    ([73.0, 25.0, 74.0, 26.0], 'raj_tile5'),
    ([77.0, 8.0,  78.0, 9.0],  'tn_tile1'),
    ([78.0, 8.0,  79.0, 9.0],  'tn_tile2'),
    ([77.0, 9.0,  78.0, 10.0], 'tn_tile3'),
    ([78.0, 9.0,  79.0, 10.0], 'tn_tile4'),
    ([79.0, 9.0,  80.0, 10.0], 'tn_tile5'),
]

for coords, name in scenes:
    print(f"\n📥 {name}...")
    os.makedirs(f'input/{name}/', exist_ok=True)

    aoi = ee.Geometry.Rectangle(coords)
    image = ee.ImageCollection('LANDSAT/LC09/C02/T1') \
        .filterDate('2024-01-01', '2024-06-30') \
        .filterBounds(aoi) \
        .sort('CLOUD_COVER') \
        .first()

    for band in ['B2','B3','B4','B10']:
        path = f'input/{name}/{name}_{band}.TIF'
        if os.path.exists(path):
            print(f"  ⏭️ {band} exists")
            continue
        try:
            geemap.ee_export_image(
                image.select([band]),
                filename=path,
                scale=100,  # 100m = smaller file size
                region=aoi,
                file_per_band=False
            )
            print(f"  ✅ {band}")
        except Exception as ex:
            print(f"  ❌ {band}: {ex}")

print("\n✅ All done!")

In [ ]:
import shutil, os

shutil.rmtree('output/', ignore_errors=True)
os.makedirs('output/', exist_ok=True)
print("✅ Cleared")

!python driver.py


In [ ]:
import os

total = 0
for root, dirs, files in os.walk('output/patches/'):
    npy = [f for f in files if f.endswith('.npy')]
    if npy:
        print(f"📁 {os.path.basename(root)}: {len(npy)}")
    total += len(npy)

print(f"\nTotal patches: {total}")
print("✅ Enough!" if total >= 200
      else "⚠️ Run more tiles")

In [ ]:
import shutil, os

base = '/content/drive/MyDrive/BAH2026'
os.makedirs(base, exist_ok=True)

# Save patches
shutil.copytree('output/',
    f'{base}/output/', dirs_exist_ok=True)

# Save raw bands
shutil.copytree('input/',
    f'{base}/input/', dirs_exist_ok=True)

print("✅ Everything saved to Drive!")

In [ ]:
# Read the file and find patch size
with open('scripts/create_patches.py', 'r') as f:
    content = f.read()

# Show lines with patch size
for i, line in enumerate(content.split('\n')):
    if '256' in line or 'patch' in line.lower() \
       or 'size' in line.lower():
        print(f"{i+1}: {line}")

In [ ]:
with open('scripts/create_patches.py', 'r') as f:
    content = f.read()

# Replace 256 with 64 everywhere
new_content = content.replace('256', '64')

with open('scripts/create_patches.py', 'w') as f:
    f.write(new_content)

print("✅ Patch size changed to 64×64")

# Verify
!grep -n "64\|patch_size" scripts/create_patches.py \
    | head -10

In [ ]:
import shutil, os
shutil.rmtree('output/', ignore_errors=True)
os.makedirs('output/', exist_ok=True)
!python driver.py

2026-06-27 05:21:37,168 - /content/IR-colorization-BAH2026/output - INFO - Processing product: gwalior_rural
2026-06-27 05:21:37,169 - /content/IR-colorization-BAH2026/output - INFO - Running: python /content/IR-colorization-BAH2026/scripts/merge_rgb.py /content/IR-colorization-BAH2026/input/gwalior_rural/gwalior_rural_B4.TIF /content/IR-colorization-BAH2026/input/gwalior_rural/gwalior_rural_B3.TIF /content/IR-colorization-BAH2026/input/gwalior_rural/gwalior_rural_B2.TIF /content/IR-colorization-BAH2026/output/rgb_images/gwalior_rural_rgb_30m.tif
2026-06-27 05:21:41,298 - /content/IR-colorization-BAH2026/output - WARNING - STDERR from merge_rgb.py:
<tifffile.TiffPage 0 @8> parsing GDAL_NODATA tag raised ValueError("invalid literal for int() with base 10: '0.0'")
<tifffile.TiffPage 0 @8> parsing GDAL_NODATA tag raised ValueError("invalid literal for int() with base 10: '0.0'")
<tifffile.TiffPage 0 @8> parsing GDAL_NODATA tag raised ValueError("invalid literal for int() with base 10: '0.

In [ ]:
import os
total = 0
for root, dirs, files in os.walk('output/patches/'):
    npy = [f for f in files if f.endswith('.npy')]
    if npy:
        print(f"📁 {os.path.basename(root)}: {len(npy)}")
    total += len(npy)
print(f"\nTotal: {total}")
print("✅ Good!" if total >= 200 else "⚠️ Need more")

📁 sample_004: 3
📁 sample_001: 3
📁 sample_005: 3
📁 sample_000: 3
📁 sample_003: 3
📁 sample_002: 3
📁 sample_004: 3
📁 sample_001: 3
📁 sample_005: 3
📁 sample_000: 3
📁 sample_003: 3
📁 sample_002: 3
📁 sample_004: 3
📁 sample_001: 3
📁 sample_005: 3
📁 sample_000: 3
📁 sample_003: 3
📁 sample_002: 3
📁 sample_004: 3
📁 sample_001: 3
📁 sample_005: 3
📁 sample_000: 3
📁 sample_003: 3
📁 sample_002: 3
📁 sample_004: 3
📁 sample_001: 3
📁 sample_005: 3
📁 sample_000: 3
📁 sample_003: 3
📁 sample_002: 3
📁 sample_004: 3
📁 sample_001: 3
📁 sample_005: 3
📁 sample_000: 3
📁 sample_003: 3
📁 sample_002: 3
📁 sample_004: 3
📁 sample_001: 3
📁 sample_005: 3
📁 sample_000: 3
📁 sample_003: 3
📁 sample_002: 3
📁 sample_004: 3
📁 sample_001: 3
📁 sample_005: 3
📁 sample_000: 3
📁 sample_003: 3
📁 sample_002: 3
📁 sample_004: 3
📁 sample_001: 3
📁 sample_005: 3
📁 sample_000: 3
📁 sample_003: 3
📁 sample_002: 3
📁 sample_000: 3
📁 sample_004: 3
📁 sample_001: 3
📁 sample_005: 3
📁 sample_000: 3
📁 sample_003: 3
📁 sample_002: 3
📁 sample_004: 3
📁 sample

In [ ]:
import shutil, os
base = '/content/drive/MyDrive/BAH2026'
os.makedirs(base, exist_ok=True)
shutil.copytree('output/', f'{base}/output/',
                dirs_exist_ok=True)
shutil.copytree('input/', f'{base}/input/',
                dirs_exist_ok=True)
print("✅ Saved to Drive!")

✅ Saved to Drive!


In [ ]:
import ee, geemap, os

ee.Initialize(project='hackathonisro2026')

# Same size as demo (Delhi) which worked well
# 0.8° × 0.6° areas at scale=30
new_scenes = [
    ([80.8, 26.7, 81.6, 27.3], 'lucknow'),
    ([85.0, 25.3, 85.8, 25.9], 'patna'),
    ([73.7, 19.8, 74.5, 20.4], 'aurangabad'),
    ([75.7, 26.8, 76.5, 27.4], 'jaipur'),
    ([76.9, 10.9, 77.7, 11.5], 'coimbatore'),
]

for coords, name in new_scenes:
    print(f"\n📥 {name}...")
    os.makedirs(f'input/{name}/', exist_ok=True)
    aoi = ee.Geometry.Rectangle(coords)

    image = ee.ImageCollection('LANDSAT/LC09/C02/T1') \
        .filterDate('2024-01-01', '2024-06-30') \
        .filterBounds(aoi) \
        .sort('CLOUD_COVER') \
        .first()

    try:
        pid = image.get('LANDSAT_PRODUCT_ID').getInfo()
        print(f"  Scene: {pid}")
    except:
        print(f"  ⚠️ No scene, skip")
        continue

    for band in ['B2','B3','B4','B10']:
        path = f'input/{name}/{name}_{band}.TIF'
        if os.path.exists(path):
            print(f"  ⏭️ {band} exists")
            continue
        try:
            geemap.ee_export_image(
                image.select([band]),
                filename=path,
                scale=30,  # 30m like demo!
                region=aoi,
                file_per_band=False
            )
            print(f"  ✅ {band}")
        except Exception as e:
            print(f"  ❌ {band}: {e}")

print("\n✅ Done!")


📥 lucknow...
  Scene: LC09_L1TP_144041_20240505_20240505_02_T1
Generating URL ...
Please wait ...
Data downloaded to /content/IR-colorization-BAH2026/input/lucknow/lucknow_B2.TIF
  ✅ B2
Generating URL ...
Please wait ...
Data downloaded to /content/IR-colorization-BAH2026/input/lucknow/lucknow_B3.TIF
  ✅ B3
Generating URL ...
Please wait ...
Data downloaded to /content/IR-colorization-BAH2026/input/lucknow/lucknow_B4.TIF
  ✅ B4
Generating URL ...
Please wait ...
Data downloaded to /content/IR-colorization-BAH2026/input/lucknow/lucknow_B10.TIF
  ✅ B10

📥 patna...
  Scene: LC09_L1TP_141042_20240430_20240430_02_T1
Generating URL ...
Please wait ...
Data downloaded to /content/IR-colorization-BAH2026/input/patna/patna_B2.TIF
  ✅ B2
Generating URL ...
Please wait ...
Data downloaded to /content/IR-colorization-BAH2026/input/patna/patna_B3.TIF
  ✅ B3
Generating URL ...
Please wait ...
Data downloaded to /content/IR-colorization-BAH2026/input/patna/patna_B4.TIF
  ✅ B4
Generating URL ...
Plea

In [ ]:
import shutil, os
shutil.rmtree('output/', ignore_errors=True)
os.makedirs('output/', exist_ok=True)
!python driver.py

2026-06-27 05:27:56,723 - /content/IR-colorization-BAH2026/output - INFO - Processing product: gwalior_rural
2026-06-27 05:27:56,724 - /content/IR-colorization-BAH2026/output - INFO - Running: python /content/IR-colorization-BAH2026/scripts/merge_rgb.py /content/IR-colorization-BAH2026/input/gwalior_rural/gwalior_rural_B4.TIF /content/IR-colorization-BAH2026/input/gwalior_rural/gwalior_rural_B3.TIF /content/IR-colorization-BAH2026/input/gwalior_rural/gwalior_rural_B2.TIF /content/IR-colorization-BAH2026/output/rgb_images/gwalior_rural_rgb_30m.tif
2026-06-27 05:27:58,741 - /content/IR-colorization-BAH2026/output - WARNING - STDERR from merge_rgb.py:
<tifffile.TiffPage 0 @8> parsing GDAL_NODATA tag raised ValueError("invalid literal for int() with base 10: '0.0'")
<tifffile.TiffPage 0 @8> parsing GDAL_NODATA tag raised ValueError("invalid literal for int() with base 10: '0.0'")
<tifffile.TiffPage 0 @8> parsing GDAL_NODATA tag raised ValueError("invalid literal for int() with base 10: '0.

In [ ]:
import ee, geemap, os

ee.Initialize(project='hackathonisro2026')

# 0.8° x 0.6° boxes at scale=30 — same recipe that gave 6 patches each
more_scenes = [
    ([75.7, 22.6, 76.5, 23.2], 'indore'),
    ([83.2, 17.6, 84.0, 18.2], 'visakhapatnam'),
    ([91.7, 26.1, 92.5, 26.7], 'guwahati'),
    ([70.7, 22.9, 71.5, 23.5], 'rajkot'),
    ([81.5, 21.0, 82.3, 21.6], 'raipur'),
]

for coords, name in more_scenes:
    print(f"\n📥 {name}...")
    os.makedirs(f'input/{name}/', exist_ok=True)
    aoi = ee.Geometry.Rectangle(coords)

    image = ee.ImageCollection('LANDSAT/LC09/C02/T1') \
        .filterDate('2024-01-01', '2024-06-30') \
        .filterBounds(aoi) \
        .sort('CLOUD_COVER') \
        .first()

    try:
        pid = image.get('LANDSAT_PRODUCT_ID').getInfo()
        print(f"  Scene: {pid}")
    except:
        print(f"  ⚠️ No scene, skipping")
        continue

    for band in ['B2','B3','B4','B10']:
        path = f'input/{name}/{name}_{band}.TIF'
        if os.path.exists(path):
            print(f"  ⏭️ {band} exists")
            continue
        try:
            geemap.ee_export_image(
                image.select([band]),
                filename=path,
                scale=30,
                region=aoi,
                file_per_band=False
            )
            print(f"  ✅ {band}")
        except Exception as e:
            print(f"  ❌ {band}: {e}")

print("\n✅ Done!")


📥 indore...
  Scene: LC09_L1TP_146044_20240503_20240503_02_T1
Generating URL ...
Please wait ...
Data downloaded to /content/IR-colorization-BAH2026/input/indore/indore_B2.TIF
  ✅ B2
Generating URL ...
Please wait ...
Data downloaded to /content/IR-colorization-BAH2026/input/indore/indore_B3.TIF
  ✅ B3
Generating URL ...
Please wait ...
Data downloaded to /content/IR-colorization-BAH2026/input/indore/indore_B4.TIF
  ✅ B4
Generating URL ...
Please wait ...
Data downloaded to /content/IR-colorization-BAH2026/input/indore/indore_B10.TIF
  ✅ B10

📥 visakhapatnam...
  Scene: LC09_L1TP_141047_20240329_20240329_02_T1
Generating URL ...
Please wait ...
Data downloaded to /content/IR-colorization-BAH2026/input/visakhapatnam/visakhapatnam_B2.TIF
  ✅ B2
Generating URL ...
Please wait ...
Data downloaded to /content/IR-colorization-BAH2026/input/visakhapatnam/visakhapatnam_B3.TIF
  ✅ B3
Generating URL ...
Please wait ...
Data downloaded to /content/IR-colorization-BAH2026/input/visakhapatnam/visa

In [ ]:
import shutil, os
shutil.rmtree('output/', ignore_errors=True)
os.makedirs('output/', exist_ok=True)
!python driver.py

2026-06-27 05:33:37,573 - /content/IR-colorization-BAH2026/output - INFO - Processing product: gwalior_rural
2026-06-27 05:33:37,575 - /content/IR-colorization-BAH2026/output - INFO - Running: python /content/IR-colorization-BAH2026/scripts/merge_rgb.py /content/IR-colorization-BAH2026/input/gwalior_rural/gwalior_rural_B4.TIF /content/IR-colorization-BAH2026/input/gwalior_rural/gwalior_rural_B3.TIF /content/IR-colorization-BAH2026/input/gwalior_rural/gwalior_rural_B2.TIF /content/IR-colorization-BAH2026/output/rgb_images/gwalior_rural_rgb_30m.tif
2026-06-27 05:33:40,100 - /content/IR-colorization-BAH2026/output - WARNING - STDERR from merge_rgb.py:
<tifffile.TiffPage 0 @8> parsing GDAL_NODATA tag raised ValueError("invalid literal for int() with base 10: '0.0'")
<tifffile.TiffPage 0 @8> parsing GDAL_NODATA tag raised ValueError("invalid literal for int() with base 10: '0.0'")
<tifffile.TiffPage 0 @8> parsing GDAL_NODATA tag raised ValueError("invalid literal for int() with base 10: '0.

In [ ]:
import ee, geemap, os

ee.Initialize(project='hackathonisro2026')

more_scenes = [
    ([77.2, 28.4, 78.0, 29.0], 'noida'),
    ([72.5, 23.0, 73.3, 23.6], 'vadodara'),
    ([80.2, 13.0, 81.0, 13.6], 'chennai'),
    ([75.3, 19.8, 76.1, 20.4], 'nashik'),
    ([85.8, 20.2, 86.6, 20.8], 'bhubaneswar'),
]

for coords, name in more_scenes:
    print(f"\n📥 {name}...")
    os.makedirs(f'input/{name}/', exist_ok=True)
    aoi = ee.Geometry.Rectangle(coords)

    image = ee.ImageCollection('LANDSAT/LC09/C02/T1') \
        .filterDate('2024-01-01', '2024-06-30') \
        .filterBounds(aoi) \
        .sort('CLOUD_COVER') \
        .first()

    try:
        pid = image.get('LANDSAT_PRODUCT_ID').getInfo()
        print(f"  Scene: {pid}")
    except:
        print(f"  ⚠️ No scene, skipping")
        continue

    for band in ['B2','B3','B4','B10']:
        path = f'input/{name}/{name}_{band}.TIF'
        if os.path.exists(path):
            print(f"  ⏭️ {band} exists")
            continue
        try:
            geemap.ee_export_image(
                image.select([band]),
                filename=path,
                scale=30,
                region=aoi,
                file_per_band=False
            )
            print(f"  ✅ {band}")
        except Exception as e:
            print(f"  ❌ {band}: {e}")

print("\n✅ Done!")


📥 noida...
  Scene: LC09_L1TP_146041_20240519_20240519_02_T1
Generating URL ...
Please wait ...
Data downloaded to /content/IR-colorization-BAH2026/input/noida/noida_B2.TIF
  ✅ B2
Generating URL ...
Please wait ...
Data downloaded to /content/IR-colorization-BAH2026/input/noida/noida_B3.TIF
  ✅ B3
Generating URL ...
Please wait ...
Data downloaded to /content/IR-colorization-BAH2026/input/noida/noida_B4.TIF
  ✅ B4
Generating URL ...
Please wait ...
Data downloaded to /content/IR-colorization-BAH2026/input/noida/noida_B10.TIF
  ✅ B10

📥 vadodara...
  Scene: LC09_L1TP_148044_20240501_20240501_02_T1
Generating URL ...
Please wait ...
Data downloaded to /content/IR-colorization-BAH2026/input/vadodara/vadodara_B2.TIF
  ✅ B2
Generating URL ...
Please wait ...
Data downloaded to /content/IR-colorization-BAH2026/input/vadodara/vadodara_B3.TIF
  ✅ B3
Generating URL ...
Please wait ...
Data downloaded to /content/IR-colorization-BAH2026/input/vadodara/vadodara_B4.TIF
  ✅ B4
Generating URL ...
P

In [ ]:
import shutil, os
shutil.rmtree('output/', ignore_errors=True)
os.makedirs('output/', exist_ok=True)
!python driver.py

2026-06-27 05:39:09,604 - /content/IR-colorization-BAH2026/output - INFO - Processing product: gwalior_rural
2026-06-27 05:39:09,605 - /content/IR-colorization-BAH2026/output - INFO - Running: python /content/IR-colorization-BAH2026/scripts/merge_rgb.py /content/IR-colorization-BAH2026/input/gwalior_rural/gwalior_rural_B4.TIF /content/IR-colorization-BAH2026/input/gwalior_rural/gwalior_rural_B3.TIF /content/IR-colorization-BAH2026/input/gwalior_rural/gwalior_rural_B2.TIF /content/IR-colorization-BAH2026/output/rgb_images/gwalior_rural_rgb_30m.tif
2026-06-27 05:39:11,376 - /content/IR-colorization-BAH2026/output - WARNING - STDERR from merge_rgb.py:
<tifffile.TiffPage 0 @8> parsing GDAL_NODATA tag raised ValueError("invalid literal for int() with base 10: '0.0'")
<tifffile.TiffPage 0 @8> parsing GDAL_NODATA tag raised ValueError("invalid literal for int() with base 10: '0.0'")
<tifffile.TiffPage 0 @8> parsing GDAL_NODATA tag raised ValueError("invalid literal for int() with base 10: '0.

In [ ]:
import os
total = 0
for root, dirs, files in os.walk('output/patches/'):
    npy = [f for f in files if f.endswith('.npy')]
    if npy:
        print(f"📁 {os.path.basename(root)}: {len(npy)}")
    total += len(npy)
print(f"\nTotal: {total}")
print("✅ Proceed to Pix2Pix!" if total >= 100 else "⚠️ Add more")

📁 sample_004: 3
📁 sample_001: 3
📁 sample_005: 3
📁 sample_000: 3
📁 sample_003: 3
📁 sample_002: 3
📁 sample_004: 3
📁 sample_001: 3
📁 sample_005: 3
📁 sample_000: 3
📁 sample_003: 3
📁 sample_002: 3
📁 sample_004: 3
📁 sample_001: 3
📁 sample_005: 3
📁 sample_000: 3
📁 sample_003: 3
📁 sample_002: 3
📁 sample_004: 3
📁 sample_001: 3
📁 sample_005: 3
📁 sample_000: 3
📁 sample_003: 3
📁 sample_002: 3
📁 sample_004: 3
📁 sample_001: 3
📁 sample_005: 3
📁 sample_000: 3
📁 sample_003: 3
📁 sample_002: 3
📁 sample_004: 3
📁 sample_001: 3
📁 sample_005: 3
📁 sample_000: 3
📁 sample_003: 3
📁 sample_002: 3
📁 sample_004: 3
📁 sample_001: 3
📁 sample_005: 3
📁 sample_000: 3
📁 sample_003: 3
📁 sample_002: 3
📁 sample_004: 3
📁 sample_001: 3
📁 sample_005: 3
📁 sample_000: 3
📁 sample_003: 3
📁 sample_002: 3
📁 sample_004: 3
📁 sample_001: 3
📁 sample_005: 3
📁 sample_000: 3
📁 sample_003: 3
📁 sample_002: 3
📁 sample_000: 3
📁 sample_004: 3
📁 sample_001: 3
📁 sample_005: 3
📁 sample_000: 3
📁 sample_003: 3
📁 sample_002: 3
📁 sample_004: 3
📁 sample

In [ ]:
import shutil, os
base = '/content/drive/MyDrive/BAH2026'
os.makedirs(base, exist_ok=True)
shutil.copytree('output/', f'{base}/output/', dirs_exist_ok=True)
shutil.copytree('input/', f'{base}/input/', dirs_exist_ok=True)
print("✅ Saved to Drive!")

✅ Saved to Drive!


In [ ]:
!git config --global user.email "sanketaher2329@gmail.com"
!git config --global user.name "DarkCipher-z"
print("✅ Git configured")

✅ Git configured


In [ ]:
import shutil, os

src = '/content/drive/MyDrive/BAH2026/tir2rgb_pipeline.ipynb'
dst = f'/content/{REPO_NAME}/tir2rgb_pipeline.ipynb'

if os.path.exists(src):
    shutil.copy(src, dst)
    print("✅ Notebook copied")
else:
    print("⚠️ Notebook not found at", src)
    print("Files in BAH2026 folder:", os.listdir('/content/drive/MyDrive/BAH2026'))

NameError: name 'REPO_NAME' is not defined

In [ ]:
os.chdir(f'/content/{REPO_NAME}')

!git config --global user.email "your@email.com"
!git config --global user.name "DarkCipher-z"
!git add .
!git commit -m "Day 1: Data download + patch generation complete"

push_result = os.system("git push origin main")
if push_result == 0:
    print("✅ Pushed to GitHub!")
else:
    print("❌ Push failed - see error above")

In [ ]:
import subprocess
result = subprocess.run(['find', '/content/drive/MyDrive', '-iname', '*tir2rgb*'], capture_output=True, text=True)
print(result.stdout)

In [ ]:
import os, shutil
from google.colab import userdata

# === Step 1: Define everything fresh (required every new session) ===
GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
GITHUB_USER  = "DarkCipher-z"
REPO_NAME    = "bah2026-ps10-team"

# === Step 2: Re-clone repo (this is gone every time the runtime resets) ===
os.chdir('/content')
if os.path.isdir(f'/content/{REPO_NAME}'):
    shutil.rmtree(f'/content/{REPO_NAME}')  # clean slate, avoid "already exists" errors

clone_result = os.system(f"git clone https://{GITHUB_TOKEN}@github.com/{GITHUB_USER}/{REPO_NAME}.git")
if not os.path.isdir(f'/content/{REPO_NAME}'):
    raise SystemExit("❌ Clone failed - check token/repo access before continuing")
print("✅ Repo cloned")

# === Step 3: Copy the notebook (confirmed real path from your Drive) ===
src = '/content/drive/MyDrive/BAH2026/ tir2rgb_pipeline.ipynb'
dst = f'/content/{REPO_NAME}/ tir2rgb_pipeline.ipynb'

if os.path.exists(src):
    shutil.copy(src, dst)
    print("✅ Notebook copied")
else:
    raise SystemExit(f"❌ Notebook still not found at {src}")

# === Step 4: Commit and push ===
os.chdir(f'/content/{REPO_NAME}')

os.system('git config --global user.email "sanketaher2329@gmail.com"')
os.system('git config --global user.name "DarkCipher-z"')
os.system('git add .')
os.system('git commit -m "Day 1: Data download + patch generation complete"')

push_result = os.system("git push origin main")
if push_result == 0:
    print("✅ Pushed to GitHub!")
else:
    print("❌ Push failed - see error above")

In [ ]:
import os
os.chdir('/content')
!git clone https://github.com/junyanz/pytorch-CycleGAN-and-pix2pix.git
os.chdir('pytorch-CycleGAN-and-pix2pix')
!pip install -r requirements.txt --quiet
print("✅ Pix2Pix ready!")

In [ ]:
import os

print("Before:", os.getcwd())

target = '/content/pytorch-CycleGAN-and-pix2pix'
if os.path.isdir(target):
    os.chdir(target)
    print("After:", os.getcwd())
    print("Files here:", os.listdir('.')[:10])
else:
    print("❌ Folder doesn't exist - clone may have gone elsewhere. Check /content:")
    print(os.listdir('/content'))

In [ ]:
!pip install -r requirements.txt --quiet
print("✅ Pix2Pix ready!")

In [ ]:
import os
nested = '/content/pytorch-CycleGAN-and-pix2pix/IR-colorization-BAH2026'
print("Contents:", os.listdir(nested))

In [ ]:
import shutil, os

src = '/content/pytorch-CycleGAN-and-pix2pix/IR-colorization-BAH2026'
dst = '/content/IR-colorization-BAH2026'

if os.path.isdir(dst):
    shutil.rmtree(dst)  # clear out the empty/incomplete one from earlier

shutil.move(src, dst)
print("✅ Moved to:", dst)
print("Contents:", os.listdir(dst))

In [ ]:
shutil.copytree('/content/drive/MyDrive/BAH2026/input/', f'{dst}/input/', dirs_exist_ok=True)
shutil.copytree('/content/drive/MyDrive/BAH2026/output/', f'{dst}/output/', dirs_exist_ok=True)
print("✅ input/ and output/ restored")

os.chdir(dst)
print("Now in:", os.getcwd())
print("Top-level contents:", os.listdir('.'))

In [ ]:
import os
total = 0
for root, dirs, files in os.walk('/content/IR-colorization-BAH2026/output/patches/'):
    npy = [f for f in files if f.endswith('.npy')]
    total += len(npy)
print(f"Total patches restored: {total}")

In [ ]:
import os
os.chdir('/content/pytorch-CycleGAN-and-pix2pix')
print("Now in:", os.getcwd())
print("Contents:", os.listdir('.'))

In [ ]:
!pip install -r requirements.txt --quiet
print("✅ Pix2Pix dependencies installed")

In [ ]:
!pip install dominate visdom wandb --quiet
print("✅ Pix2Pix dependencies installed")

In [ ]:
!cat environment.yml

In [ ]:
import numpy as np
from PIL import Image
import os

os.makedirs('datasets/tir2rgb/train', exist_ok=True)
os.makedirs('datasets/tir2rgb/val', exist_ok=True)

patch_base = '/content/IR-colorization-BAH2026/output/patches/'
count = 0

for scene in os.listdir(patch_base):
    scene_path = os.path.join(patch_base, scene)
    if not os.path.isdir(scene_path):
        continue
    for sample in os.listdir(scene_path):
        sp = os.path.join(scene_path, sample)
        if not os.path.isdir(sp):
            continue
        files = os.listdir(sp)
        tir_f = next((f for f in files if 'tir_200m' in f and f.endswith('.npy')), None)
        rgb_f = next((f for f in files if 'rgb' in f and f.endswith('.npy')), None)
        if not tir_f or not rgb_f:
            continue

        tir = np.load(os.path.join(sp, tir_f))
        rgb = np.load(os.path.join(sp, rgb_f))

        def norm(x):
            x = x.astype(np.float32)
            mn, mx = x.min(), x.max()
            if mx - mn < 1e-8:
                return np.zeros_like(x, dtype=np.uint8)
            return ((x - mn) / (mx - mn) * 255).astype(np.uint8)

        tir_n = norm(tir.squeeze())
        tir_rgb = np.stack([tir_n]*3, axis=-1)

        if rgb.ndim == 3 and rgb.shape[0] == 3:
            rgb_img = np.transpose(rgb, (1,2,0))
        else:
            rgb_img = rgb
        rgb_img = norm(rgb_img)
        if rgb_img.ndim == 2:
            rgb_img = np.stack([rgb_img]*3, axis=-1)

        tir_pil = Image.fromarray(tir_rgb).resize((256,256))
        rgb_pil = Image.fromarray(rgb_img).resize((256,256))

        combined = Image.new('RGB', (512, 256))
        combined.paste(tir_pil, (0, 0))
        combined.paste(rgb_pil, (256, 0))

        folder = 'val' if count % 10 == 0 else 'train'
        combined.save(f'datasets/tir2rgb/{folder}/{count:04d}.png')
        count += 1

print(f"✅ {count} training pairs created!")
print(f"Train: {len(os.listdir('datasets/tir2rgb/train'))} | Val: {len(os.listdir('datasets/tir2rgb/val'))}")